# Section 08 — Hierarchy Semantic Labeling

This notebook adds semantic descriptions to the temporally coupled hierarchy
generated in Section 07.

The goal is not to modify the hierarchy structure, but to annotate existing
supernodes with:

- human-readable labels;
- one-sentence glosses;
- supporting evidence terms;
- confidence estimates.

The hierarchy topology, node memberships, hyperedges, and persistent identities
remain unchanged.

Input:

Section 07 temporal hierarchy

Output:

Labelled hierarchy with semantic interpretations.

# 08_labels

In [1]:
from pathlib import Path

REPO_DIR = Path(
    "/content/tkh-hierarchy-project"
)

if not REPO_DIR.exists():
    !git clone https://github.com/mohamadghoroobi/tkh-hierarchy-project.git

%cd /content/tkh-hierarchy-project

Cloning into 'tkh-hierarchy-project'...
remote: Enumerating objects: 75, done.
remote: Counting objects: 100% (75/75), done.
remote: Compressing objects: 100% (58/58), done.
remote: Total 75 (delta 29), reused 57 (delta 15), pack-reused 0 (from 0)
Receiving objects: 100% (75/75), 16.85 MiB | 40.32 MiB/s, done.
Resolving deltas: 100% (29/29), done.
/content/tkh-hierarchy-project


## Imports

In [2]:
import json
import re
from pathlib import Path
from collections import Counter, defaultdict

import pandas as pd
import numpy as np

## Paths

In [48]:
TEMPORAL_DIR = (
    PROJECT_DIR
    /
    "artifacts"
    /
    "hierarchy"
    /
    "temporal"
)


LABEL_DIR = (
    PROJECT_DIR
    /
    "artifacts"
    /
    "hierarchy"
    /
    "labelled"
)


LABEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)


SNAPSHOT_YEARS = [
    2020,
    2022,
    2024,
    2026
]


LABEL_LEVELS = [
    0,
    1
]

## Load TKH metadata

In [49]:
TKH_PATH = (
    PROJECT_DIR
    /
    "data"
    /
    "tkh_collection10.json"
)


with open(
    TKH_PATH,
    "r",
    encoding="utf-8"
) as f:

    tkh = json.load(f)


nodes = tkh["nodes"]


node_lookup = {
    node["id"]: node
    for node in nodes
}


print(
    "Loaded nodes:",
    len(node_lookup)
)

Loaded nodes: 5798


## Load temporal hierarchy

In [50]:
def load_temporal_hierarchy(year):

    path = (
        TEMPORAL_DIR
        /
        f"hierarchy_{year}_temporal_unlabelled.json"
    )


    with open(
        path,
        "r",
        encoding="utf-8"
    ) as f:

        hierarchy = json.load(f)


    print(
        year,
        "| P0:",
        len(hierarchy["levels"]["0"]),
        "| P1:",
        len(hierarchy["levels"]["1"]),
        "| P2:",
        len(hierarchy["levels"]["2"])
    )


    return hierarchy

## Load all snapshots

In [51]:
hierarchies = {

    year:
        load_temporal_hierarchy(year)

    for year
    in SNAPSHOT_YEARS
}

2020 | P0: 12 | P1: 60 | P2: 300
2022 | P0: 12 | P1: 60 | P2: 300
2024 | P0: 12 | P1: 60 | P2: 300
2026 | P0: 12 | P1: 60 | P2: 300


## Validation before labeling

In [52]:
for year in SNAPSHOT_YEARS:

    assert len(
        hierarchies[year]["levels"]["0"]
    ) > 0

    assert len(
        hierarchies[year]["levels"]["1"]
    ) > 0


print(
    "Hierarchy validation passed."
)

Hierarchy validation passed.


## Extract surface forms

In [53]:
def member_surface_forms(member_ids):

    forms = []


    for node_id in member_ids:

        node = node_lookup.get(
            node_id
        )

        if node is None:
            continue


        text = (
            node.get(
                "surface_form"
            )
            or
            node.get(
                "name"
            )
            or
            ""
        )


        if text:
            forms.append(text)


    return forms

## Candidate terms

In [54]:
def normalize_term(text):

    text = text.lower()

    text = re.sub(
        r"[^a-z0-9\s\-]",
        "",
        text
    )

    return text.strip()

In [55]:
def candidate_terms(
    forms,
    top_k=10
):

    counter = Counter()


    for form in forms:

        for token in form.split():

            token = normalize_term(
                token
            )


            if len(token) > 3:

                counter[token] += 1


    return [
        term
        for term, _
        in counter.most_common(top_k)
    ]

## Label generation

In [56]:
def generate_label(
    terms
):

    if len(terms) == 0:

        return "Unresolved concept"


    return terms[0].title()

## Gloss generation

In [57]:
def generate_gloss(
    label,
    terms
):

    evidence = ", ".join(
        terms[:5]
    )


    return (
        f"This concept represents a cluster "
        f"associated with {label.lower()}, "
        f"supported by evidence terms: {evidence}."
    )

## Confidence

In [58]:
def label_confidence(
    terms,
    size
):

    if size == 0:

        return 0.0


    evidence = min(
        len(terms)/5,
        1.0
    )


    size_factor = min(
        np.log1p(size)/5,
        1.0
    )


    return float(
        0.5*evidence
        +
        0.5*size_factor
    )

## Main labeling function

In [59]:
def label_hierarchy(
    hierarchy,
    year
):

    labelled = copy.deepcopy(
        hierarchy
    )


    labelled["label_metadata"] = {

        "year":
            year,

        "method":
            "evidence-based lexical labeling",

        "future_information_used":
            False
    }



    for level in LABEL_LEVELS:


        original_records = (
            hierarchy["levels"][str(level)]
        )


        labelled["levels"][str(level)] = []



        for record in original_records:


            new_record = copy.deepcopy(
                record
            )


            member_ids = (
                record["member_ids"]
            )


            forms = member_surface_forms(
                member_ids
            )


            terms = candidate_terms(
                forms
            )


            new_record["label"] = (
                generate_label(
                    terms
                )
            )


            new_record["gloss"] = (
                generate_gloss(
                    new_record["label"],
                    terms
                )
            )


            new_record["label_evidence"] = terms


            new_record["label_confidence"] = (
                label_confidence(
                    terms,
                    len(member_ids)
                )
            )


            labelled["levels"][str(level)].append(
                new_record
            )


    return labelled

## Apply labeling

In [60]:
labelled_hierarchies = {}


for year in SNAPSHOT_YEARS:

    labelled_hierarchies[year] = (
        label_hierarchy(
            hierarchies[year],
            year
        )
    )

## Verify structure unchanged

In [61]:
for year in SNAPSHOT_YEARS:

    print(
        year,
        "P0:",
        len(
            hierarchies[year]["levels"]["0"]
        ),
        "→",
        len(
            labelled_hierarchies[year]["levels"]["0"]
        ),

        "| P1:",
        len(
            hierarchies[year]["levels"]["1"]
        ),
        "→",
        len(
            labelled_hierarchies[year]["levels"]["1"]
        )
    )

2020 P0: 12 → 12 | P1: 60 → 60
2022 P0: 12 → 12 | P1: 60 → 60
2024 P0: 12 → 12 | P1: 60 → 60
2026 P0: 12 → 12 | P1: 60 → 60


## Inspect labels

In [62]:
for year in SNAPSHOT_YEARS:

    print(
        "\nYEAR:",
        year
    )


    for record in (
        labelled_hierarchies[year]
        ["levels"]["0"][:5]
    ):

        print(
            record["label"],
            "→",
            record["gloss"]
        )


YEAR: 2020
Prediction → This concept represents a cluster associated with prediction, supported by evidence terms: prediction, with, energy, materials, training.
Materials → This concept represents a cluster associated with materials, supported by evidence terms: materials, calculation, database, calculations, aflow.
Carbon → This concept represents a cluster associated with carbon, supported by evidence terms: carbon, amorphous, potential, surface, potentials.
Wang → This concept represents a cluster associated with wang, supported by evidence terms: wang, chen, david, hart, marco.
Magpie → This concept represents a cluster associated with magpie, supported by evidence terms: magpie, matbench, coulomb, matrix, random.

YEAR: 2022
Prediction → This concept represents a cluster associated with prediction, supported by evidence terms: prediction, with, materials, energy, neural.
Md-17 → This concept represents a cluster associated with md-17, supported by evidence terms: md-17, original

## Export labelled hierarchy

In [63]:
for year, hierarchy in labelled_hierarchies.items():

    path = (
        LABEL_DIR
        /
        f"hierarchy_{year}_labelled.json"
    )


    with open(
        path,
        "w",
        encoding="utf-8"
    ) as f:

        json.dump(
            hierarchy,
            f,
            indent=2
        )


    print(
        "Saved:",
        path
    )

Saved: /content/tkh-hierarchy-project/artifacts/hierarchy/labelled/hierarchy_2020_labelled.json
Saved: /content/tkh-hierarchy-project/artifacts/hierarchy/labelled/hierarchy_2022_labelled.json
Saved: /content/tkh-hierarchy-project/artifacts/hierarchy/labelled/hierarchy_2024_labelled.json
Saved: /content/tkh-hierarchy-project/artifacts/hierarchy/labelled/hierarchy_2026_labelled.json


## Create label table

In [64]:
rows = []


for year, hierarchy in labelled_hierarchies.items():

    for level in LABEL_LEVELS:

        for record in hierarchy["levels"][str(level)]:

            rows.append({

                "year":
                    year,

                "level":
                    level,

                "persistent_id":
                    record["persistent_id"],

                "label":
                    record["label"],

                "confidence":
                    record["label_confidence"]
            })


label_df = pd.DataFrame(rows)


label_df.head()

,year,level,persistent_id,label,confidence
0,2020,0,L0_C00000,Prediction,1.000000
1,2020,0,L0_C00001,Materials,1.000000
2,2020,0,L0_C00002,Carbon,1.000000
3,2020,0,L0_C00003,Wang,0.951086
4,2020,0,L0_C00004,Magpie,0.961512


## Save table

In [65]:
label_df.to_csv(
    LABEL_DIR /
    "label_table.csv",
    index=False
)


print(
    "Label table saved."
)

Label table saved.


## Final summary

In [66]:
print(
    "===== SECTION 08 COMPLETE ====="
)

print(
    "Snapshots:",
    SNAPSHOT_YEARS
)

print(
    "Labelled levels:",
    LABEL_LEVELS
)

print(
    "Total labelled supernodes:",
    len(label_df)
)

===== SECTION 08 COMPLETE =====
Snapshots: [2020, 2022, 2024, 2026]
Labelled levels: [0, 1]
Total labelled supernodes: 288


## Git Push